# Silver Layer Transformations

## Objectives
1. **Data Quality**: Fix data types, handle nulls, remove duplicates
2. **Standardization**: Clean column names, standardize values
3. **Enrichment**: Join dimensions and add business context
4. **Business Logic**: Add calculated fields and validation flags
5. **Audit**: Add metadata and processing timestamps

## Source Tables
- `workspace.bronze.crm_sales_details` - Sales transactions
- `workspace.bronze.crm_cust_info` - Customer information
- `workspace.bronze.crm_prd_info` - Product information
- `workspace.bronze.erp_px_cat_g1v2` - Product categories

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from datetime import datetime

## Step 1: Read Bronze Layer Tables

In [0]:
# Read bronze tables
df_sales_bronze = spark.table("workspace.bronze.crm_sales_details")
df_customer_bronze = spark.table("workspace.bronze.crm_cust_info")
df_product_bronze = spark.table("workspace.bronze.crm_prd_info")
df_product_category = spark.table("workspace.bronze.erp_px_cat_g1v2")

print(f"Sales records: {df_sales_bronze.count()}")
print(f"Customer records: {df_customer_bronze.count()}")
print(f"Product records: {df_product_bronze.count()}")
print(f"Category records: {df_product_category.count()}")

## Step 2: Transform Sales Data
- Convert INT dates to proper DATE types
- Convert financial fields to DECIMAL
- Clean column names
- Handle nulls

In [0]:
# Transform sales data
df_sales_silver = df_sales_bronze \
    .withColumnRenamed("sls_ord_num", "order_number") \
    .withColumnRenamed("sls_prd_key", "product_key") \
    .withColumnRenamed("sls_cust_id", "customer_id") \
    .withColumnRenamed("sls_quantity", "quantity") \
    .withColumn("order_date", F.to_date(F.col("sls_order_dt").cast("string"), "yyyyMMdd")) \
    .withColumn("ship_date", F.to_date(F.col("sls_ship_dt").cast("string"), "yyyyMMdd")) \
    .withColumn("due_date", F.to_date(F.col("sls_due_dt").cast("string"), "yyyyMMdd")) \
    .withColumn("sales_amount", F.col("sls_sales").cast(DecimalType(18, 2))) \
    .withColumn("unit_price", F.col("sls_price").cast(DecimalType(18, 2))) \
    .drop("sls_order_dt", "sls_ship_dt", "sls_due_dt", "sls_sales", "sls_price")

# Add calculated fields
df_sales_silver = df_sales_silver \
    .withColumn("calculated_revenue", F.col("quantity") * F.col("unit_price")) \
    .withColumn("shipping_delay_days", F.datediff(F.col("ship_date"), F.col("order_date"))) \
    .withColumn("delivery_status", 
                F.when(F.col("ship_date") <= F.col("due_date"), "On Time")
                 .when(F.col("ship_date").isNull(), "Not Shipped")
                 .otherwise("Delayed"))

# Data quality flags
df_sales_silver = df_sales_silver \
    .withColumn("is_valid",
                F.when((F.col("quantity") > 0) & 
                       (F.col("unit_price") > 0) & 
                       (F.col("order_date").isNotNull()) &
                       (F.col("customer_id").isNotNull()) &
                       (F.col("product_key").isNotNull()), True)
                 .otherwise(False))

display(df_sales_silver.limit(5))

## Step 3: Transform Customer Dimension
- Clean column names
- Standardize categorical values
- Remove duplicates
- Add customer age calculation

In [0]:
# Transform customer dimension
df_customer_silver = df_customer_bronze \
    .withColumnRenamed("cst_id", "customer_id") \
    .withColumnRenamed("cst_key", "customer_key") \
    .withColumnRenamed("cst_firstname", "first_name") \
    .withColumnRenamed("cst_lastname", "last_name") \
    .withColumnRenamed("cst_marital_status", "marital_status") \
    .withColumnRenamed("cst_gndr", "gender") \
    .withColumnRenamed("cst_create_date", "customer_since_date")

# Standardize values and trim whitespace
df_customer_silver = df_customer_silver \
    .withColumn("first_name", F.trim(F.col("first_name"))) \
    .withColumn("last_name", F.trim(F.col("last_name"))) \
    .withColumn("gender", F.upper(F.trim(F.col("gender")))) \
    .withColumn("marital_status", F.upper(F.trim(F.col("marital_status"))))

# Add calculated fields
df_customer_silver = df_customer_silver \
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name"))) \
    .withColumn("customer_tenure_days", F.datediff(F.current_date(), F.col("customer_since_date")))

# Remove duplicates based on customer_id
df_customer_silver = df_customer_silver.dropDuplicates(["customer_id"])

display(df_customer_silver.limit(5))

## Step 4: Transform Product Dimension
- Clean column names
- Add product category information
- Calculate product active status

In [0]:
# Transform product dimension
df_product_silver = df_product_bronze \
    .withColumnRenamed("prd_id", "product_id") \
    .withColumnRenamed("prd_key", "product_key") \
    .withColumnRenamed("prd_nm", "product_name") \
    .withColumnRenamed("prd_cost", "product_cost") \
    .withColumnRenamed("prd_line", "product_line") \
    .withColumnRenamed("prd_start_dt", "product_start_date") \
    .withColumnRenamed("prd_end_dt", "product_end_date")

# Clean product category dimension
df_category_silver = df_product_category \
    .withColumnRenamed("ID", "category_id") \
    .withColumnRenamed("CAT", "category") \
    .withColumnRenamed("SUBCAT", "subcategory") \
    .withColumnRenamed("MAINTENANCE", "requires_maintenance")

# Join product with category based on product_key prefix match
df_product_silver = df_product_silver \
    .join(df_category_silver, 
          df_product_silver.product_key.startswith(df_category_silver.category_id),
          "left")

# Add calculated fields
df_product_silver = df_product_silver \
    .withColumn("product_cost_decimal", F.col("product_cost").cast(DecimalType(18, 2))) \
    .withColumn("is_active", 
                F.when((F.col("product_start_date") <= F.current_date()) &
                       ((F.col("product_end_date") >= F.current_date()) | F.col("product_end_date").isNull()),
                       True)
                 .otherwise(False)) \
    .drop("product_cost")

display(df_product_silver.limit(5))

## Step 5: Create Enriched Sales Fact Table
- Join sales with customer and product dimensions
- Add audit metadata

In [0]:
# Join sales with customer dimension
df_sales_enriched = df_sales_silver \
    .join(df_customer_silver, "customer_id", "left") \
    .join(df_product_silver, "product_key", "left")

# Add audit columns
df_sales_enriched = df_sales_enriched \
    .withColumn("processed_timestamp", F.current_timestamp()) \
    .withColumn("source_system", F.lit("CRM")) \
    .withColumn("data_quality_score",
                F.when(F.col("is_valid") & F.col("customer_id").isNotNull() & F.col("category").isNotNull(), "HIGH")
                 .when(F.col("is_valid"), "MEDIUM")
                 .otherwise("LOW"))

# Select final columns in logical order
df_sales_fact_silver = df_sales_enriched.select(
    # Order information
    "order_number",
    "order_date",
    "ship_date",
    "due_date",
    "shipping_delay_days",
    "delivery_status",
    
    # Customer information
    "customer_id",
    "customer_key",
    "full_name",
    "gender",
    "marital_status",
    "customer_since_date",
    "customer_tenure_days",
    
    # Product information
    "product_key",
    "product_id",
    "product_name",
    "product_line",
    "category",
    "subcategory",
    "requires_maintenance",
    "is_active",
    
    # Financial metrics
    "quantity",
    "unit_price",
    "sales_amount",
    "calculated_revenue",
    "product_cost_decimal",
    
    # Quality and audit
    "is_valid",
    "data_quality_score",
    "source_system",
    "processed_timestamp"
)

print(f"Total enriched records: {df_sales_fact_silver.count()}")
print(f"Valid records: {df_sales_fact_silver.filter(F.col('is_valid') == True).count()}")
display(df_sales_fact_silver.limit(10))

## Step 6: Write to Silver Layer Tables

In [0]:
# Write fact table to silver layer
df_sales_fact_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.sales_fact")

# Write customer dimension
df_customer_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.customer_dim")

# Write product dimension
df_product_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.product_dim")

print("✓ Silver layer tables created successfully!")

## Step 7: Data Quality Summary

In [0]:
# Data quality summary
print("=" * 60)
print("SILVER LAYER DATA QUALITY SUMMARY")
print("=" * 60)

# Sales fact summary
sales_summary = spark.table("workspace.silver.sales_fact")
print(f"\n📊 Sales Fact Table:")
print(f"  Total Records: {sales_summary.count():,}")
print(f"  Valid Records: {sales_summary.filter(F.col('is_valid')).count():,}")
print(f"  High Quality: {sales_summary.filter(F.col('data_quality_score') == 'HIGH').count():,}")
print(f"  Date Range: {sales_summary.agg(F.min('order_date')).collect()[0][0]} to {sales_summary.agg(F.max('order_date')).collect()[0][0]}")
print(f"  Total Revenue: ${sales_summary.agg(F.sum('calculated_revenue')).collect()[0][0]:,.2f}")

# Customer dimension summary
cust_summary = spark.table("workspace.silver.customer_dim")
print(f"\n👥 Customer Dimension:")
print(f"  Total Customers: {cust_summary.count():,}")
print(f"  Avg Tenure: {cust_summary.agg(F.avg('customer_tenure_days')).collect()[0][0]:.0f} days")

# Product dimension summary
prod_summary = spark.table("workspace.silver.product_dim")
print(f"\n📦 Product Dimension:")
print(f"  Total Products: {prod_summary.count():,}")
print(f"  Active Products: {prod_summary.filter(F.col('is_active')).count():,}")
print(f"  Categories: {prod_summary.select('category').distinct().count()}")

print("\n" + "=" * 60)
print("✓ Silver layer transformation completed!")
print("=" * 60)

In [0]:
%sql
-- Quick validation: Sales by category
SELECT 
  category,
  subcategory,
  COUNT(*) as order_count,
  SUM(quantity) as total_quantity,
  ROUND(SUM(calculated_revenue), 2) as total_revenue,
  ROUND(AVG(unit_price), 2) as avg_price
FROM workspace.silver.sales_fact
WHERE is_valid = true
GROUP BY category, subcategory
ORDER BY total_revenue DESC
LIMIT 10